# HPA Data Access with `pyVO` 

```
Author: ESDC Team at ESAC
Date Last Modified: 20/08/2026
```

This notebook demonstrate programmatic access to the data in ESA's [HPA](https://hpa.esa.int) using the Table Access Protocol (TAP) and the Astronomical Query Language (ADQL) with the [PyVO](https://pyvo.readthedocs.io/en/latest/) `python` package. Please refer to the [PyVO documentation](https://pyvo.readthedocs.io/en/stable/dal/index.html#pyvo-scs) for a more in-depth introduction.

## Requirements

The following python packages are required to run this notebook: `pyvo`, `parfive`. You can install them by running the line below in your terminal (after removing the leading `#`).

In [1]:
# python -m pip install pyvo parfive

In [2]:
from pathlib import Path
import urllib.parse

import parfive
import pyvo as vo

/Users/max.mahlke/apps/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Connecting to the HPA via TAP

To connect to the HPA, we use the TAP service protocol. This requires a TAP service URL, which is provided by the HPA in the documentation pages of the different archives. We use here the URL of the Proba-2 mission archive.

In [3]:
URL_TAP = "https://p2sa.esac.esa.int/p2sa-sl-tap/tap"

service = vo.dal.TAPService(URL_TAP)

## Inspecting table metadata

The Table Acess Protocol exposes different tables to the user. We can inspect the tables present in a serivce using the `tables` attribute of the `TAPService` object. Each `table` attribute further contains information on the columns and datatypes it provides.

In [4]:
table_names = [table.name for table in service.tables]
print(len(service.tables), "tables found in the TAP service: ", table_names)

21 tables found in the TAP service:  ['p2sa.file', 'p2sa.full_disk_solar_image', 'p2sa.instrument', 'p2sa.lyra_observation', 'p2sa.observation', 'p2sa.observatory', 'p2sa.science_object', 'p2sa.swap_observation', 'p2sa.v_carrington_rotation_file', 'p2sa.v_file', 'p2sa.v_lyra_observation', 'p2sa.v_observation', 'p2sa.v_swap_observation', 'public.dual', 'tap_config.coord_sys', 'tap_config.properties', 'tap_schema.columns', 'tap_schema.key_columns', 'tap_schema.keys', 'tap_schema.schemas', 'tap_schema.tables']


To inspect the columns of a specific table, you can use the following code:

In [5]:
table_observation = service.tables['p2sa.observation']

for column in table_observation.columns:
    print(f"  {column.name} ({column.datatype.content})")

  begin_date (char)
  calibrated (boolean)
  end_date (char)
  file_format (char)
  file_name (char)
  file_path (char)
  file_size (long)
  instrument_oid (int)
  observation_oid (int)
  observation_type (char)
  processing_level (char)
  science_objective (char)
  science_object_oid (int)
  wavelength_range (char)


This table contains metadata of Proba-2 observations, including start and end dates, calibration levels, filenames, and instrument IDs. In the P2Sa, there is also a `v_observation` table. To see how it differs, we compare the columns between the two tables.  

In [6]:
columns_observation = [column.name for column in service.tables['p2sa.observation'].columns]
columns_v_observation = [column.name for column in service.tables['p2sa.v_observation'].columns]

difference = set(columns_v_observation) - set(columns_observation)
print("Columns in p2sa.v_observation but not in p2sa.observation:", difference)

Columns in p2sa.v_observation but not in p2sa.observation: {'observatory_name', 'instrument_name', 'science_object_name'}



We see that `v_observation` is slightly more human-readable than `observation`, as it contains the instrument, observatory, and science object names as well as their numeric IDs.

## Accessing table data

We now want to get the content of the tables themselves. This can be achieved via *synchronous* or *asynchronous* AQDL queries. The PyVO documentation contains a nice [summary of the pros and cons](https://pyvo.readthedocs.io/en/stable/dal/index.html#synchronous-vs-asynchronous-query) of each, but in short:

- Synchronous queries return the result immediately and generally do not require a login. They are best for small (in terms of data volume) queries.
- Asynchronous queries first compute and prepare the query result on the remote (HPA) side, before providing the user with a download link. These queries support much larger data volume but may require you to log in to an archive user account prior to querying.

### Synchronous table data access

All data queries are written in ADQL syntax. Extensive documentation for this query language is available [here](https://www.ivoa.net/documents/ADQL/20180112/PR-ADQL-2.1-20180112.html). To run a synchronous query with `pyVO`, use the `search` method of the `TAPService` object.

In [7]:
results = service.search("select top 100 * from p2sa.observation")
print(results)

<DALResultsTable length=100>
       begin_date       calibrated ... science_object_oid wavelength_range
         object            bool    ...       int32             object     
----------------------- ---------- ... ------------------ ----------------
2019-10-13T07:07:28.296      False ...                  1              174
2018-10-11T05:30:53.648       True ...                  1              174
2019-10-13T05:50:28.163      False ...                  1              174
2018-10-11T04:24:53.634       True ...                  1              174
2020-10-14T19:23:32.265      False ...                  1              174
2018-10-11T03:48:13.543       True ...                  1              174
2019-10-13T03:07:17.988      False ...                  1              174
2019-10-13T02:26:57.996      False ...                  1              174
2018-10-11T02:01:53.483       True ...                  1              174
                    ...        ... ...                ...              

This list contains the first 100 entries of the Proba-2 science archive observations table. You can convert it from the custom `DALResultsTable` type to the more common `Astropy.Table` or `Pandas.DataFrame` types.

In [8]:
results = results.to_table() # convert the results to an astropy table
results = results.to_pandas() # convert the astropy table to a pandas dataframe

Some services limit the number of rows that are returned in a single synchronous query. These limits are available via the `maxrec` and `hardlimit` attributes of the `TAPService`. `maxrec` can be changed by the user (refer to the [PyVO documentation](https://pyvo.readthedocs.io/en/stable/dal/index.html#query-limit)), but it cannot exceed `hardlimit`. For the P2SA, there are no limits in place, hence, looking up the attributes raises an error:

In [9]:
try:
    print("The set record limit is:", service.maxrec)
except:
    print("The set record limit is not available for this TAP service.")

try:
    print("The maximum record limit is:", service.hardlimit)
except:
    print("The maximum record limit is not available for this TAP service.")

The set record limit is not available for this TAP service.
The maximum record limit is not available for this TAP service.


Even though there is no limit in place, requesting the table of all observations in the `p2sa.observations` table via a synchronous query will likely not work, as the server connection will be lost before the query can finish. Large queries need to be run asynchronously.

### Asynchronous table data access

The `run_async` method of the `TAPService` object allows to execute ADQL queries asynchronously with almost the same syntax as for synchronous queries (only replacing `search` with `run_async`).

In [10]:
# This query takes about 5 minutes to complete, so it is commented out for now.
# You can uncomment it to run it if you want to retrieve all the records from
# the p2sa.observation table.

# results = tap_service.run_async("select * from p2sa.observation")
# print(results)

Executing the query above returned 7475989 total observations and finished in about 5 minutes.


Under the hood, `run_async` submits a job to the TAP service, queries the job state intermittently, and retrieves the result once the job is completed. All these steps can also be executed manually using `pyVO`, as documented [here](https://pyvo.readthedocs.io/en/stable/dal/index.html#synchronous-vs-asynchronous-query). This gives a greater degree of flexbility (e.g. by providing progress information) by is slightly more verbose. 

## Accessing data products

Now we want to download the actual files behind each row in a metadata table. To do this, we send the same ADQL queries that we constructed above to the `data/` endpoints of the HPA archives, which in turn return the files that the ADQL queries produce. The query results need to contain the `file_name` column, which the HPA archives use to identify the files to return.
We use the `parfive` package to perform the downloads.

We start by buidling the query for the files we are interested in (in this case, simply the first 5 files in the `v_observation` table) and use `pyVO` to verify the resulting filelist.

In [11]:
query = """
    select top 5 * from p2sa.v_observation 
"""

result = service.search(query)
result = result.to_table() # convert the results to an astropy table
print(result[['begin_date', 'instrument_name', 'file_name']])

       begin_date       instrument_name           file_name          
----------------------- --------------- -----------------------------
2026-08-06T08:39:38.752            SWAP swap_lv0_20260806_083938.fits
2026-08-06T09:23:38.794            SWAP swap_lv0_20260806_092338.fits
2026-08-05T08:09:17.161            SWAP swap_lv1_20260805_080917.fits
2026-08-05T08:53:17.211            SWAP swap_lv1_20260805_085317.fits
2026-08-05T10:26:47.308            SWAP swap_lv1_20260805_102647.fits


With the list of files we are interested in, we download them using two helper functions below: 

- `download_files`: Perform the TAP query and pass the list of requested files (their `file_name`s) to a `parfive.Downloader`. The files are downloaded and stored in the `output_directory`.

- `build_download_url`: Build the URL that retrieves the file from the HPA archive.

In [12]:
def build_download_url(filename):
    URL_DATA = "https://p2sa.esac.esa.int/p2sa-sl-tap/data"
    query = f"SELECT file_path, file_name FROM p2sa.v_observation WHERE file_name='{filename}'"
    return URL_DATA + "?retrieval_type=PRODUCT&QUERY=" + urllib.parse.quote(query)

def download_files(query, output_directory):
    URL_TAP = "https://p2sa.esac.esa.int/p2sa-sl-tap/tap"
    service = vo.dal.TAPService(URL_TAP)

    result = service.search(query)
    print(f"Returned {len(result)} rows")

    dl = parfive.Downloader()
    outdir = Path(output_directory)

    for row in result:
        filename = row["file_name"]
        url = build_download_url(filename)

        dl.enqueue_file(url, path=outdir)

    print("Starting download…")
    files = dl.download()
    
    if files.errors:
        for _, url, exc in files.errors:
            print(f"{exc}")

download_files(query, output_directory="downloads")

Returned 5 rows
Starting download…


Files Downloaded: 100%|██████████| 5/5 [00:00<00:00, 17.19file/s]


The downloaded files are in the newly created `downloads/` directory.